In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report
)

# Baca dataset
df = pd.read_csv("dataset_prediksi_sampah_150_lengkap.csv")

# Periksa data
print("Ukuran dataset:", df.shape)
print(df.head())
print("\nJumlah data kosong:\n", df.isnull().sum())

# Fitur dan target
fitur = [
    "Jenis Sampah",
    "Jumlah Sampah (kg)",
    "Lokasi",
    "Status Pengangkutan"
]
target = "Prediksi Sampah"

X = df[fitur]
y = df[target]

# Pisahkan data: 80% latih dan 20% uji
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Ubah fitur kategori menjadi angka dan skalakan jumlah sampah
fitur_kategori = ["Jenis Sampah", "Lokasi", "Status Pengangkutan"]
fitur_numerik = ["Jumlah Sampah (kg)"]

preprocessing = ColumnTransformer([
    ("kategori", OneHotEncoder(handle_unknown="ignore", sparse_output=False), fitur_kategori),
    ("numerik", StandardScaler(), fitur_numerik)
])

# Model yang diuji
model = {
    "K-NN K=3": KNeighborsClassifier(n_neighbors=3),
    "K-NN K=4": KNeighborsClassifier(n_neighbors=4),
    "K-NN K=5": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB()
}

hasil = []

for nama, algoritma in model.items():
    pipeline = Pipeline([
        ("preprocessing", preprocessing),
        ("model", algoritma)
    ])

    # Latih model
    pipeline.fit(X_train, y_train)

    # Prediksi data uji
    prediksi = pipeline.predict(X_test)

    # Evaluasi
    akurasi = accuracy_score(y_test, prediksi)

    # 5-fold cross-validation pada data latih
    skor_cv = cross_val_score(
        pipeline, X_train, y_train,
        cv=5, scoring="accuracy"
    )

    hasil.append({
        "Model": nama,
        "Akurasi Data Uji": akurasi,
        "Rata-rata Akurasi CV 5-Fold": skor_cv.mean()
    })

    print(f"\n===== {nama} =====")
    print("Akurasi:", round(akurasi, 3))
    print("Confusion Matrix:\n", confusion_matrix(y_test, prediksi))
    print("\nClassification Report:\n",
          classification_report(y_test, prediksi, zero_division=0))

    # Bandingkan akurasi data latih dan data uji untuk melihat indikasi overfitting
    akurasi_latih = pipeline.score(X_train, y_train)
    print("Akurasi data latih:", round(akurasi_latih, 3))
    print("Akurasi data uji:", round(akurasi, 3))

# Ringkasan hasil semua model
tabel_hasil = pd.DataFrame(hasil)
print("\nRingkasan hasil:")
print(tabel_hasil)

Ukuran dataset: (150, 6)
   No Jenis Sampah  Jumlah Sampah (kg)       Lokasi Status Pengangkutan  \
0   1      Organik                   4       Oesapa      Belum diangkut   
1   2      Plastik                  29  Kelapa Lima      Sudah diangkut   
2   3        Logam                  12         Alak      Belum diangkut   
3   4      Organik                   4       Oebobo      Sudah diangkut   
4   5      Plastik                  65         Alak      Sudah diangkut   

  Prediksi Sampah  
0          Rendah  
1          Rendah  
2          Rendah  
3          Rendah  
4          Sedang  

Jumlah data kosong:
 No                     0
Jenis Sampah           0
Jumlah Sampah (kg)     0
Lokasi                 0
Status Pengangkutan    0
Prediksi Sampah        0
dtype: int64

===== K-NN K=3 =====
Akurasi: 0.8
Confusion Matrix:
 [[ 8  2  0]
 [ 1 10  2]
 [ 0  1  6]]

Classification Report:
               precision    recall  f1-score   support

      Rendah       0.89      0.80      0.84     